In [11]:
import sys
print(sys.executable)

e:\TuttiQuanti\Trabajo\Anaconda3\envs\env_LLMzCor\python.exe


## Puesta a punto

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

1. Definir las rutas de los archivos

In [2]:
# 1) Ruta al .txt con tus PMIDs (uno por línea)
ruta_txt = r"E:\PaperLLM\LLMzCor.github.io\WebScraping\PMIDs\PMID_Ct1.txt"

# 2) Directorio donde guardar los CSV con los Abstracts
carpeta_salida = r"E:\PaperLLM\LLMzCor.github.io\WebScraping\Abstracts"

2. Función para cargar PMIDs desde un archivo .txt

In [4]:
def load_pmids_from_txt(path_txt: str, encoding: str = "utf-8") -> list[str]:
    """
    Lee un archivo .txt que contiene un PMID por línea y
    devuelve una lista de PMIDs sin líneas vacías.
    """
    with open(path_txt, "r", encoding=encoding) as f:
        return [line.strip() for line in f if line.strip()]


3. Función para obtener el abstract de PubMed

In [5]:
def fetch_abstract(pmid: str) -> str:
    """
    Hace GET a la página de PubMed para el PMID dado,
    parsea el HTML con BeautifulSoup y extrae el abstract.
    """
    url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
    resp = requests.get(url)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    cont = soup.find("div", {"class": "abstract-content selected"}) \
           or soup.find("div", {"class": "abstract"})
    return cont.get_text(" ", strip=True) if cont else ""


4. Función principal (orquestación del flujo)

In [6]:
def main():
    # 1) Ruta al .txt con tus PMIDs (Definidas en la sección 1)
    pmids = load_pmids_from_txt(ruta_txt)

    records = []
    for pmid in pmids:
        try:
            abstract = fetch_abstract(pmid)
        except Exception as e:
            print(f"[Error] PMID {pmid}: {e}")
            abstract = ""
        records.append({"PMID": pmid, "Abstract": abstract})
        time.sleep(1.0)  # pausa de 1s entre peticiones para no saturar PubMed

    # 2) Crear DataFrame y exportar a CSV
    df = pd.DataFrame(records)
    df.to_csv("pmids_abstracts.csv", index=False, encoding="utf-8-sig")
    print(f"Procesados {len(records)} PMIDs. Resultado en pmids_abstracts.csv")


5. Punto de entrada del script

In [7]:
if __name__ == "__main__":
    import os

    # 1) Ruta al .txt con tus PMIDs (uno por línea)
    pmids = load_pmids_from_txt(ruta_txt)

    # 2) Directorio donde guardar los CSV
    os.makedirs(carpeta_salida, exist_ok=True)

    # 3) Nombre base extraído del .txt (sin extensión)
    base = os.path.splitext(os.path.basename(ruta_txt))[0]

    records = []      # irá acumulando cada {"PMID","Abstract"}
    completed = False # marcamos si el loop terminó sin lanzar excepción

    try:
        for pmid in pmids:
            try:
                abstract = fetch_abstract(pmid)
            except Exception as e:
                print(f"[Error] PMID {pmid}: {e}")
                abstract = ""
            records.append({"PMID": pmid, "Abstract": abstract})
            time.sleep(1.0)  # pausa para no saturar PubMed
        completed = True
    except Exception as e:
        print(f"[Error crítico] {e}. Se guardarán los resultados parciales.")
    finally:
        # 4) Creamos el DataFrame con lo que haya en 'records'
        df = pd.DataFrame(records)

        # 5) Definimos el nombre de archivo según si completó o no
        if completed:
            nombre_csv = f"{base}.csv"
        else:
            nombre_csv = f"{base}_partial.csv"

        # 6) Ruta completa de salida y guardado
        salida = os.path.join(carpeta_salida, nombre_csv)
        df.to_csv(salida, index=False, encoding="utf-8-sig")
        print(f"{len(records)} registros guardados en '{salida}'.")


30 registros guardados en 'E:\PaperLLM\LLMzCor.github.io\WebScraping\Abstracts\PMID_Ct1.csv'.
